In [ ]:
import sys
sys.path.append('..')

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from simulation.create_network import create_microgrid
from simulation.run_powerflow import run_powerflow_analysis, check_constraints
from simulation.topology_switching import TopologySwitcher
from simulation.visualization import plot_network, plot_results, plot_constraints

np.random.seed(42)

## 1. Network Visualization

In [ ]:
# Create different networks
mesh_graph, _ = create_microgrid(num_nodes=10, topology='mesh', seed=42)
ring_graph, _ = create_microgrid(num_nodes=10, topology='ring', seed=42)
tree_graph, _ = create_microgrid(num_nodes=10, topology='tree', seed=42)

# Visualize all three
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

graphs = [('Mesh', mesh_graph, 'spring'), ('Ring', ring_graph, 'circular'), ('Tree', tree_graph, 'spring')]

for idx, (name, graph, layout) in enumerate(graphs):
    ax = axes[idx]
    
    # Position nodes
    if layout == 'circular':
        pos = nx.circular_layout(graph)
    else:
        pos = nx.spring_layout(graph, k=2, iterations=50, seed=42)
    
    # Color nodes
    node_colors = []
    for node in graph.nodes():
        gen = graph.nodes[node].get('generation', 0)
        if gen > 50:
            node_colors.append('red')
        elif gen > 0:
            node_colors.append('orange')
        else:
            node_colors.append('lightblue')
    
    # Draw
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=500, ax=ax, alpha=0.8)
    nx.draw_networkx_edges(graph, pos, width=2, ax=ax, alpha=0.6)
    nx.draw_networkx_labels(graph, pos, font_size=9, ax=ax)
    
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2. Power Flow Visualization

In [ ]:
# Run power flow analysis
results = run_powerflow_analysis(mesh_graph)

# Plot results
fig, axes = plot_results(results, figsize=(15, 10))
plt.tight_layout()
plt.show()

In [ ]:
# Check constraints
violations = check_constraints(mesh_graph, results)

print(f"Constraint Violations:")
print(f"  Voltage violations: {len(violations['voltage_violations'])}")
print(f"  Thermal violations: {len(violations['thermal_violations'])}")
print(f"  Frequency violations: {len(violations['frequency_violations'])}")

# Plot violations
fig, axes = plot_constraints(violations, figsize=(14, 5))
plt.tight_layout()
plt.show()

## 3. Topology Switching

In [ ]:
# Initialize topology switcher
switcher = TopologySwitcher(mesh_graph)

print(f"Initial topology: {switcher.current_graph.number_of_edges()} edges")
print(f"Base topology: {switcher.base_graph.number_of_edges()} edges")

In [ ]:
# Get possible reconfigurations
reconfigs = switcher.get_possible_reconfigurations()
print(f"Possible reconfigurations: {len(reconfigs)}")

# Show first few
for i, reconfig in enumerate(reconfigs[:3]):
    print(f"  Config {i+1}: {reconfig.number_of_edges()} edges")

In [ ]:
# Switch to a reconfiguration with fewer edges (more efficient)
optimal_topo = switcher.optimize_for_losses()
print(f"\nOptimal topology found: {optimal_topo.number_of_edges()} edges")
print(f"Current topology: {switcher.current_graph.number_of_edges()} edges")

# Switch to optimal
success = switcher.switch_to_topology(optimal_topo)
print(f"Switch successful: {success}")

In [ ]:
# Compare power flow before and after switching
results_before = run_powerflow_analysis(mesh_graph)
results_after = run_powerflow_analysis(switcher.current_graph)

print(f"\nPower Flow Comparison:")
print(f"Before switch:")
print(f"  Edges: {mesh_graph.number_of_edges()}")
print(f"  Total Loss: {results_before['total_loss']:.6f} MW")
print(f"\nAfter switch:")
print(f"  Edges: {switcher.current_graph.number_of_edges()}")
print(f"  Total Loss: {results_after['total_loss']:.6f} MW")
print(f"\nImprovement: {(results_before['total_loss'] - results_after['total_loss']) / results_before['total_loss'] * 100:.2f}%")

## 4. Fault Handling

In [ ]:
# Reset to base topology
switcher.reset_to_base()
print(f"Reset to base topology: {switcher.current_graph.number_of_edges()} edges")

# Simulate a line outage
failed_edge = list(switcher.current_graph.edges())[0]
print(f"\nFailed edge: {failed_edge}")

# Try to island after fault
islanding_success = switcher.island_on_fault(failed_edge)
print(f"Islanding successful: {islanding_success}")
print(f"Network after islanding: {switcher.current_graph.number_of_nodes()} nodes, {switcher.current_graph.number_of_edges()} edges")

In [ ]:
# Restore the network
restoration_success = switcher.restore_after_fault([failed_edge])
print(f"Restoration successful: {restoration_success}")
print(f"Network after restoration: {switcher.current_graph.number_of_nodes()} nodes, {switcher.current_graph.number_of_edges()} edges")

In [ ]:
# View switch history
history = switcher.get_switch_history()
print(f"\nSwitch History ({len(history)} switches):")
for i, switch in enumerate(history):
    print(f"\nSwitch {i+1}:")
    print(f"  Edges removed: {len(switch['edges_removed'])}")
    print(f"  Edges added: {len(switch['edges_added'])}")

## 5. Summary Statistics

In [ ]:
# Summary comparison
print(f"\n{'='*50}")
print(f"MICROGRID ANALYSIS SUMMARY")
print(f"{'='*50}\n")

for name, graph in [('Mesh', mesh_graph), ('Ring', ring_graph), ('Tree', tree_graph)]:
    results = run_powerflow_analysis(graph)
    violations = check_constraints(graph, results)
    
    print(f"{name} Topology:")
    print(f"  Nodes: {graph.number_of_nodes()}, Edges: {graph.number_of_edges()}")
    print(f"  Density: {nx.density(graph):.3f}")
    print(f"  Avg Voltage: {np.mean(results['voltages']):.4f} pu")
    print(f"  Total Loss: {results['total_loss']:.6f} MW")
    print(f"  Violations: {len(violations['voltage_violations']) + len(violations['thermal_violations'])}")
    print()